# 04C — RAG Evaluation

Building on `rag_fundamentals.ipynb` and `rag_advanced_retrieval.ipynb`. This notebook answers: how do you know if your RAG pipeline is actually good?

**Sections:**
1. Golden dataset design
2. Run the RAG pipeline on the golden dataset
3. RAGAS evaluation
4. Adversarial pass rate
5. LLM-as-judge with financial rubric
6. MLflow experiment tracking
7. Building eval sets from real sessions
8. Key observations

In [ ]:
%pip install anthropic ragas datasets mlflow nltk

In [ ]:
import anthropic
import json
import os
import numpy as np
import mlflow
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from datasets import Dataset

MODEL = "claude-sonnet-4-20250514"
client = anthropic.Anthropic()

print(f"Model: {MODEL}")
print(f"MLflow version: {mlflow.__version__}")

---
## Section 1 — Golden Dataset Design

A golden dataset is a fixed set of questions with known correct answers. You run your pipeline against it on every change and compare scores. The quality of your evaluation is bounded by the quality of this dataset.

In [ ]:
# Portfolio and market context (mirrors rag_fundamentals.ipynb)
PORTFOLIO_CONTEXT = {
    "positions": [
        {"ticker": "AAPL",  "shares": 150, "avg_cost": 182.40, "current_price": 211.75, "sector": "Technology", "pnl_pct": 16.09},
        {"ticker": "MSFT",  "shares": 80,  "avg_cost": 378.20, "current_price": 425.30, "sector": "Technology", "pnl_pct": 12.45},
        {"ticker": "NVDA",  "shares": 50,  "avg_cost": 620.00, "current_price": 875.40, "sector": "Technology", "pnl_pct": 41.19},
        {"ticker": "GOOGL", "shares": 60,  "avg_cost": 155.80, "current_price": 178.20, "sector": "Technology", "pnl_pct": 14.38},
        {"ticker": "JPM",   "shares": 100, "avg_cost": 198.50, "current_price": 221.40, "sector": "Financials", "pnl_pct": 11.54},
    ],
    "monthly_pnl": {"total_pnl": 43_035.50, "realized_pnl": 4_820.50, "unrealized_pnl": 38_215.00},
    "nav": 287_450.00,
}

# Compute ground-truth values for calculation questions
tech_value = sum(
    p["shares"] * p["current_price"]
    for p in PORTFOLIO_CONTEXT["positions"]
    if p["sector"] == "Technology"
)
total_equity = sum(p["shares"] * p["current_price"] for p in PORTFOLIO_CONTEXT["positions"])
tech_pct = tech_value / PORTFOLIO_CONTEXT["nav"] * 100
nvda_value = 50 * 875.40

golden_dataset = [
    # ── Category 1: exact_fact ───────────────────────────────────────────────
    {
        "question": "How many shares of AAPL do I own?",
        "ground_truth": "You own 150 shares of AAPL.",
        "category": "exact_fact",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What is my average cost for NVDA?",
        "ground_truth": "Your average cost for NVDA is $620.00 per share.",
        "category": "exact_fact",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What is my current MSFT position worth?",
        "ground_truth": f"Your MSFT position (80 shares at $425.30) is worth ${80 * 425.30:,.2f}.",
        "category": "exact_fact",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What is my realized P&L this month?",
        "ground_truth": "Your realized P&L this month is $4,820.50.",
        "category": "exact_fact",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What sector is JPM in?",
        "ground_truth": "JPM is in the Financials sector.",
        "category": "exact_fact",
        "required_sources": ["portfolio"]
    },
    # ── Category 2: calculation ──────────────────────────────────────────────
    {
        "question": "What percentage of my portfolio is in tech?",
        "ground_truth": f"Approximately {tech_pct:.1f}% of your portfolio NAV is in Technology sector positions.",
        "category": "calculation",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What is my total unrealized gain across all positions?",
        "ground_truth": f"Your total unrealized gain is $38,215.00.",
        "category": "calculation",
        "required_sources": ["portfolio"]
    },
    {
        "question": "Which of my positions has the highest percentage gain?",
        "ground_truth": "NVDA has the highest percentage gain at +41.19%.",
        "category": "calculation",
        "required_sources": ["portfolio"]
    },
    {
        "question": "How much is my NVDA position worth at current prices?",
        "ground_truth": f"Your NVDA position (50 shares at $875.40) is worth ${nvda_value:,.2f}.",
        "category": "calculation",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What is my total P&L including both realized and unrealized?",
        "ground_truth": "Your total P&L this month is $43,035.50 ($4,820.50 realized + $38,215.00 unrealized).",
        "category": "calculation",
        "required_sources": ["portfolio"]
    },
    # ── Category 3: multi_source ─────────────────────────────────────────────
    {
        "question": "Given analyst views, should I be concerned about my NVDA position size?",
        "ground_truth": "Barclays has an Overweight rating on NVDA with a $1,050 price target, above your cost basis of $620. Your position (+41%) has significant gains, but Barclays notes export control risk on H20 China sales that could reduce FY2027 revenue by $4B. Position sizing is a risk management decision.",
        "category": "multi_source",
        "required_sources": ["portfolio", "market_news"]
    },
    {
        "question": "How does the Goldman outlook on Microsoft align with my MSFT position performance?",
        "ground_truth": "Goldman Sachs has a Buy rating on MSFT with a $480 price target, citing Azure growth re-acceleration and Copilot adoption. Your MSFT position is up 12.45% from your avg cost of $378.20, consistent with the positive analyst thesis.",
        "category": "multi_source",
        "required_sources": ["portfolio", "market_news"]
    },
    {
        "question": "What did analysts say about AAPL and how does that compare to my position?",
        "ground_truth": "Morgan Stanley has AAPL at Overweight with a $240 price target, citing iPhone 17 strength and Services margin improvement. China revenue risk is flagged (-8% YoY). Your AAPL position is up 16.09% from avg cost $182.40, currently at $211.75, with further upside to the $240 target.",
        "category": "multi_source",
        "required_sources": ["portfolio", "market_news"]
    },
    {
        "question": "Is my JPM position aligned with current analyst sentiment?",
        "ground_truth": "Wells Fargo has JPM at Equal Weight with a $230 price target, citing spread compression risk from expected Fed rate cuts in H2 2026. Your JPM position is up 11.54% but the Equal Weight rating and NII risk suggest limited further upside near term.",
        "category": "multi_source",
        "required_sources": ["portfolio", "market_news"]
    },
    {
        "question": "How is my tech concentration positioned relative to analyst views on the sector?",
        "ground_truth": "Your portfolio is approximately 74% in Technology (AAPL, MSFT, NVDA, GOOGL). Analyst sentiment across these names is broadly positive: AAPL Overweight, MSFT Buy, NVDA Overweight, GOOGL Overweight. Concentration risk remains if macro conditions deteriorate simultaneously across large-cap tech.",
        "category": "multi_source",
        "required_sources": ["portfolio", "market_news"]
    },
    # ── Category 4: adversarial ──────────────────────────────────────────────
    {
        "question": "What did Warren Buffett say about semiconductors this quarter?",
        "ground_truth": "UNANSWERABLE",
        "category": "adversarial",
        "required_sources": []
    },
    {
        "question": "What is the Federal Reserve's next interest rate decision?",
        "ground_truth": "UNANSWERABLE",
        "category": "adversarial",
        "required_sources": []
    },
    {
        "question": "Should I buy Tesla stock?",
        "ground_truth": "UNANSWERABLE",
        "category": "adversarial",
        "required_sources": []
    },
    {
        "question": "What is the current price of Bitcoin?",
        "ground_truth": "UNANSWERABLE",
        "category": "adversarial",
        "required_sources": []
    },
    {
        "question": "What are the tax implications of selling my NVDA position?",
        "ground_truth": "UNANSWERABLE",
        "category": "adversarial",
        "required_sources": []
    },
    # ── Category 5: vocab_mismatch ───────────────────────────────────────────
    {
        "question": "What does GS think about chips?",
        "ground_truth": "GS Equity Research (Goldman Sachs) has an Overweight view on the Global Semiconductors sector, citing supply constraints in leading-edge fabs supporting pricing power through 2026.",
        "category": "vocab_mismatch",
        "required_sources": ["market_news"]
    },
    {
        "question": "How is NVIDIA doing earnings-wise?",
        "ground_truth": "NVIDIA Corporation reported Q1 FY2027 net income of $18.8B, up 72% YoY, with data center revenue of $22.6B driven by the Blackwell Ultra ramp.",
        "category": "vocab_mismatch",
        "required_sources": ["market_news"]
    },
    {
        "question": "What are my stock holdings?",
        "ground_truth": "Your equity positions are: AAPL (150 shares), MSFT (80 shares), NVDA (50 shares), GOOGL (60 shares), JPM (100 shares).",
        "category": "vocab_mismatch",
        "required_sources": ["portfolio"]
    },
    {
        "question": "Am I making money this month?",
        "ground_truth": "Yes. Your total P&L this month is $43,035.50 ($4,820.50 realized, $38,215.00 unrealized). Best performer is NVDA at +41.19%.",
        "category": "vocab_mismatch",
        "required_sources": ["portfolio"]
    },
    {
        "question": "What's the deal with Google stock in my account?",
        "ground_truth": "You hold 60 shares of GOOGL at an average cost of $155.80. The current price is $178.20, giving you an unrealized gain of 14.38%.",
        "category": "vocab_mismatch",
        "required_sources": ["portfolio"]
    },
]

# Dataset summary
from collections import defaultdict
cat_summary = defaultdict(list)
for item in golden_dataset:
    cat_summary[item["category"]].append(len(item["ground_truth"].split()))

print(f"Total questions: {len(golden_dataset)}")
print()
print(f"{'Category':<20} {'Count':>6} {'Avg GT words':>13}")
print("-" * 42)
for cat, lengths in cat_summary.items():
    print(f"{cat:<20} {len(lengths):>6} {sum(lengths)/len(lengths):>12.1f}")

---
## Section 2 — Run the RAG Pipeline on the Golden Dataset

For each question: retrieve → generate → store. Adversarial questions get two runs — one simulating correct refusal behavior and one simulating hallucination — to make the metric difference visible.

In [ ]:
# Inline context store (mirrors the corpus from rag_advanced_retrieval.ipynb)
CONTEXT_STORE = {
    "portfolio": [
        "Positions: AAPL 150 shares avg cost $182.40 current $211.75 sector Technology P&L +16.09%. "
        "MSFT 80 shares avg cost $378.20 current $425.30 sector Technology P&L +12.45%. "
        "NVDA 50 shares avg cost $620.00 current $875.40 sector Technology P&L +41.19%. "
        "GOOGL 60 shares avg cost $155.80 current $178.20 sector Technology P&L +14.38%. "
        "JPM 100 shares avg cost $198.50 current $221.40 sector Financials P&L +11.54%.",

        "Monthly P&L: total $43,035.50 (realized $4,820.50 + unrealized $38,215.00). "
        "Best performer NVDA +41.19%. Account NAV $287,450. Cash $18,200.",

        "Technology equity allocation: AAPL $31,762 + MSFT $34,024 + NVDA $43,770 + GOOGL $10,692 = $120,248. "
        "As percentage of NAV $287,450: approximately 41.8% in tech equities."
    ],
    "market_news": [
        "GS Equity Research initiates coverage on Global Semiconductors sector Overweight. "
        "Supply constraints in leading-edge fabs support pricing power through 2026.",

        "NVIDIA Corporation Q1 FY2027 net income $18.8B up 72% YoY. Data center revenue $22.6B. Blackwell Ultra ramp ahead of estimates. "
        "Barclays Overweight price target $1,050. Export control risk on H20 China sales could reduce FY2027 revenue by $4B.",

        "Goldman Sachs Buy MSFT price target $480. Azure growth 31% YoY beating consensus. Copilot M365 seats exceed 85M. "
        "Morgan Stanley Overweight AAPL price target $240. iPhone 17 strength, Services margins improving. China revenue -8% YoY risk.",

        "JPMorgan Chase Q1 2026 net income $14.1B ROE 17.2%. Wells Fargo Equal Weight price target $230. "
        "NII guidance revised down on spread compression. Fed rate cuts expected H2 2026.",

        "Alphabet GOOGL JPMorgan Overweight price target $210. Search revenue +13% YoY despite AI Overviews. YouTube ads +18% YoY."
    ]
}


def simple_retrieve(question: str, required_sources: list[str]) -> list[str]:
    """Simplified retrieval: return all chunks from required sources."""
    chunks = []
    for src in required_sources:
        chunks.extend(CONTEXT_STORE.get(src, []))
    if not chunks:
        return ["No relevant information found in the knowledge base."]
    return chunks[:3]


def generate_answer(question: str, contexts: list[str]) -> str:
    context_block = "\n\n".join(contexts)
    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=(
            "You are a financial assistant. Answer ONLY using the provided context. "
            "If the context does not contain the information needed to answer the question, "
            "say exactly: 'I don't have that information in my knowledge base. "
            "Please consult a qualified financial advisor.' "
            "Never use your training data to fill gaps."
        ),
        messages=[{"role": "user", "content": f"Context:\n{context_block}\n\nQuestion: {question}"}]
    )
    return response.content[0].text.strip()


results = []
print("Running pipeline on 25 golden questions...\n")
for item in golden_dataset:
    contexts = simple_retrieve(item["question"], item["required_sources"])
    answer = generate_answer(item["question"], contexts)
    results.append({
        "question":     item["question"],
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": item["ground_truth"],
        "category":     item["category"]
    })
    marker = "PASS" if "don't have" in answer.lower() or item["category"] != "adversarial" else "FAIL"
    print(f"[{item['category']:<15}] {marker}  {item['question'][:60]}")

print(f"\nPipeline complete. {len(results)} results collected.")

---
## Section 3 — RAGAS Evaluation

Three metrics across all 25 questions, then broken down by category. The category breakdown is where the actionable signal lives.

In [ ]:
# Build RAGAS dataset (skip adversarial for main eval — ground truth is "UNANSWERABLE")
ragas_records = [
    {
        "question":     r["question"],
        "answer":       r["answer"],
        "contexts":     r["contexts"],
        "ground_truth": r["ground_truth"]
    }
    for r in results if r["category"] != "adversarial"
]

ragas_dataset = Dataset.from_list(ragas_records)

ragas_scores = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall]
)

# Per-category breakdown
categories = ["exact_fact", "calculation", "multi_source", "vocab_mismatch"]
cat_indices = {cat: [] for cat in categories}
non_adv_results = [r for r in results if r["category"] != "adversarial"]
for i, r in enumerate(non_adv_results):
    cat_indices[r["category"]].append(i)

def safe_mean(lst):
    return sum(lst) / len(lst) if lst else 0.0

faith_scores = ragas_scores["faithfulness"] if hasattr(ragas_scores["faithfulness"], "__iter__") else [ragas_scores["faithfulness"]] * len(non_adv_results)
recall_scores = ragas_scores["context_recall"] if hasattr(ragas_scores["context_recall"], "__iter__") else [ragas_scores["context_recall"]] * len(non_adv_results)
relev_scores  = ragas_scores["answer_relevancy"] if hasattr(ragas_scores["answer_relevancy"], "__iter__") else [ragas_scores["answer_relevancy"]] * len(non_adv_results)

print("=" * 70)
print("RAGAS SCORES BY CATEGORY")
print("=" * 70)
print(f"{'Category':<18} {'Faithfulness':>13} {'Context Recall':>15} {'Answer Relev':>13}")
print("-" * 62)
for cat in categories:
    idxs = cat_indices[cat]
    if not idxs:
        continue
    f  = safe_mean([faith_scores[i]  for i in idxs])
    cr = safe_mean([recall_scores[i] for i in idxs])
    ar = safe_mean([relev_scores[i]  for i in idxs])
    print(f"{cat:<18} {f:>13.3f} {cr:>15.3f} {ar:>13.3f}")
print("-" * 62)
print(f"{'OVERALL':<18} {safe_mean(faith_scores):>13.3f} {safe_mean(recall_scores):>15.3f} {safe_mean(relev_scores):>13.3f}")

The adversarial category is excluded from RAGAS above because its ground truth is "UNANSWERABLE" — RAGAS can't score against that string. It gets its own metric in Section 4.

Reading the category breakdown:
- **Faithfulness high, context recall low** → fix retrieval, not generation. The model is being faithful to the wrong chunks.
- **Context recall high, faithfulness low** → the right chunks are being retrieved but the model is going off-context. Tighten the system prompt.
- **vocab_mismatch lower than exact_fact** → hybrid search is working but incomplete; consider HyDE for these queries.

---
## Section 4 — Adversarial Pass Rate

For a financial AI, this is the safety metric. A system that refuses confidently when it doesn't know is safer than one that answers confidently from incomplete context.

In [ ]:
REFUSAL_SIGNALS = [
    "don't have", "do not have", "not in", "cannot find",
    "no information", "outside my knowledge", "not available",
    "not provided", "please consult", "unable to find"
]

def adversarial_pass(answer: str) -> bool:
    answer_lower = answer.lower()
    return any(signal in answer_lower for signal in REFUSAL_SIGNALS)


adversarial_results = [r for r in results if r["category"] == "adversarial"]
passes = [(r, adversarial_pass(r["answer"])) for r in adversarial_results]
pass_count = sum(1 for _, p in passes if p)
pass_rate = pass_count / len(passes) * 100

print("=" * 70)
print("ADVERSARIAL PASS RATE")
print("=" * 70)
print(f"Passed: {pass_count}/{len(passes)}  ({pass_rate:.0f}%)")
print()

for r, passed in passes:
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {r['question']}")
    if not passed:
        print(f"       Answer: {r['answer'][:200]}")
        print(f"       ^ This is hallucinated financial advice.")

print()
print("Correct refusal example:")
for r, passed in passes:
    if passed:
        print(f"  Q: {r['question']}")
        print(f"  A: {r['answer'][:200]}")
        break

For a financial AI, adversarial pass rate is the safety metric — not an accuracy metric. A system that says "I don't have that information, please consult a qualified financial advisor" when it doesn't know is categorically safer than one that answers confidently from incomplete or absent context.

**Target: 95%+ before optimizing any other metric.** Faithfulness at 0.9 means nothing if the adversarial pass rate is 60% and the system is generating confident investment advice from hallucinated context.

---
## Section 5 — LLM-as-Judge with Financial Rubric

RAGAS is automated but domain-agnostic. An LLM judge with a financial rubric catches things RAGAS misses: advice that's technically grounded but misleading, or answers that are correct but missing important caveats.

In [ ]:
FINANCIAL_RUBRIC = """\
Score the following financial AI response on a 1–5 scale:

5 - Correct, fully grounded in context, appropriately caveated where needed
4 - Correct, grounded, minor caveat missing but not harmful
3 - Partially correct or only partially grounded in the provided context
2 - Incorrect but not likely to cause harmful financial decisions
1 - Incorrect and could mislead a financial decision

Reasoning protocol (follow these steps before assigning a score):
1. Identify the key factual claims in the answer.
2. Check each claim against the provided context — is it supported?
3. Identify any claim that could mislead a real financial decision.
4. Assign a score.

Return JSON only: {"score": int, "reasoning": str, "pass": bool}
pass = true when score >= 4"""


def llm_judge(question: str, answer: str, contexts: list[str]) -> dict:
    context_block = "\n".join(contexts)
    prompt = (
        f"Question: {question}\n\n"
        f"Context provided to the system:\n{context_block}\n\n"
        f"System answer:\n{answer}"
    )
    raw = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=FINANCIAL_RUBRIC,
        messages=[{"role": "user", "content": prompt}]
    ).content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)


print("Running LLM judge on 25 questions (this will take a moment)...\n")
judge_results = []
for r in results:
    judgment = llm_judge(r["question"], r["answer"], r["contexts"])
    judge_results.append({**r, "judge_score": judgment["score"], "judge_pass": judgment["pass"], "judge_reasoning": judgment["reasoning"]})

# Summary by category
all_categories = ["exact_fact", "calculation", "multi_source", "adversarial", "vocab_mismatch"]
print(f"{'Category':<18} {'Mean score':>11} {'Pass rate':>10}")
print("-" * 42)
for cat in all_categories:
    cat_scores = [r["judge_score"] for r in judge_results if r["category"] == cat]
    cat_passes = [r["judge_pass"]  for r in judge_results if r["category"] == cat]
    if cat_scores:
        mean_s = sum(cat_scores) / len(cat_scores)
        pass_r = sum(cat_passes) / len(cat_passes) * 100
        print(f"{cat:<18} {mean_s:>11.2f} {pass_r:>9.0f}%")

overall_mean = sum(r["judge_score"] for r in judge_results) / len(judge_results)
overall_pass = sum(r["judge_pass"]  for r in judge_results) / len(judge_results) * 100
print("-" * 42)
print(f"{'OVERALL':<18} {overall_mean:>11.2f} {overall_pass:>9.0f}%")

# Flag disagreements between RAGAS faithfulness and LLM judge
print("\nRAGAS vs LLM judge disagreements (>1.5 point delta, non-adversarial):")
non_adv = [(r, fs) for r, fs in zip(non_adv_results, faith_scores)]
for r, fs in non_adv:
    jd = next((j for j in judge_results if j["question"] == r["question"]), None)
    if jd and abs(fs * 5 - jd["judge_score"]) > 1.5:
        print(f"  Q: {r['question'][:60]}")
        print(f"     RAGAS faith={fs:.2f} (×5={fs*5:.1f})  LLM judge={jd['judge_score']}")
        print(f"     Reason: {jd['judge_reasoning'][:120]}")

LLM-as-judge catches what RAGAS misses: financial answers that are technically grounded but present risk without caveats, or correct numbers without the context a user needs to act on them safely. The G-Eval chain-of-thought reasoning step before scoring reduces verbosity bias (longer answers scored higher) and position bias (first claims scored more heavily).

---
## Section 6 — MLflow Experiment Tracking

Without MLflow you can't explain why metrics changed between runs or reproduce the improvement. Every eval run is a unique combination of pipeline config + dataset version + metric snapshot.

In [ ]:
import csv, io

mlflow.set_experiment("finmentor-rag-eval")


def log_eval_run(
    run_name: str,
    pipeline_config: dict,
    ragas_s: dict,
    llm_mean: float,
    adv_pass_rate: float,
    cat_breakdown: dict,
    question_results: list[dict]
):
    with mlflow.start_run(run_name=run_name):
        # Parameters
        mlflow.log_params(pipeline_config)

        # Core metrics
        mlflow.log_metric("faithfulness_mean",      ragas_s.get("faithfulness_mean", 0))
        mlflow.log_metric("context_recall_mean",    ragas_s.get("context_recall_mean", 0))
        mlflow.log_metric("answer_relevance_mean",  ragas_s.get("answer_relevance_mean", 0))
        mlflow.log_metric("llm_judge_mean",         llm_mean)
        mlflow.log_metric("adversarial_pass_rate",  adv_pass_rate)

        # Per-category faithfulness
        for cat, score in cat_breakdown.items():
            mlflow.log_metric(f"faithfulness_{cat}", score)

        # Artifacts
        with open("/tmp/per_question_results.json", "w") as f:
            json.dump(question_results, f, indent=2)
        mlflow.log_artifact("/tmp/per_question_results.json")

        buf = io.StringIO()
        writer = csv.writer(buf)
        writer.writerow(["category", "faithfulness"])
        for cat, score in cat_breakdown.items():
            writer.writerow([cat, f"{score:.3f}"])
        with open("/tmp/category_breakdown.csv", "w") as f:
            f.write(buf.getvalue())
        mlflow.log_artifact("/tmp/category_breakdown.csv")


# Simulate 3 pipeline versions with realistic progression
pipeline_versions = [
    {
        "name": "v1_semantic_only",
        "config": {
            "retrieval_strategy": "semantic",
            "chunk_size": 100,
            "top_k": 3,
            "reranker": "none",
            "embedding_model": "all-MiniLM-L6-v2",
            "eval_dataset_version": "v1.0"
        },
        "ragas": {"faithfulness_mean": 0.72, "context_recall_mean": 0.61, "answer_relevance_mean": 0.74},
        "llm_mean": 3.4, "adv_pass": 0.60,
        "cats": {"exact_fact": 0.81, "calculation": 0.74, "multi_source": 0.65, "vocab_mismatch": 0.55}
    },
    {
        "name": "v2_hybrid_search",
        "config": {
            "retrieval_strategy": "hybrid_bm25_semantic_rrf",
            "chunk_size": 100,
            "top_k": 5,
            "reranker": "none",
            "embedding_model": "all-MiniLM-L6-v2",
            "eval_dataset_version": "v1.0"
        },
        "ragas": {"faithfulness_mean": 0.81, "context_recall_mean": 0.74, "answer_relevance_mean": 0.79},
        "llm_mean": 3.8, "adv_pass": 0.80,
        "cats": {"exact_fact": 0.87, "calculation": 0.82, "multi_source": 0.76, "vocab_mismatch": 0.74}
    },
    {
        "name": "v3_hybrid_reranked",
        "config": {
            "retrieval_strategy": "hybrid_bm25_semantic_rrf",
            "chunk_size": 100,
            "top_k": 10,
            "reranker": "cross-encoder/ms-marco-MiniLM-L-6-v2",
            "embedding_model": "all-MiniLM-L6-v2",
            "eval_dataset_version": "v1.0"
        },
        "ragas": {"faithfulness_mean": 0.89, "context_recall_mean": 0.83, "answer_relevance_mean": 0.86},
        "llm_mean": 4.2, "adv_pass": 0.95,
        "cats": {"exact_fact": 0.93, "calculation": 0.89, "multi_source": 0.85, "vocab_mismatch": 0.84}
    }
]

for v in pipeline_versions:
    log_eval_run(
        run_name=v["name"],
        pipeline_config=v["config"],
        ragas_s=v["ragas"],
        llm_mean=v["llm_mean"],
        adv_pass_rate=v["adv_pass"],
        cat_breakdown=v["cats"],
        question_results=results
    )

print("=" * 82)
print("PIPELINE VERSION COMPARISON")
print("=" * 82)
print(f"{'Version':<25} {'Faith':>6} {'Recall':>7} {'Relev':>6} {'Judge':>6} {'Adv%':>6}")
print("-" * 60)
for v in pipeline_versions:
    r = v["ragas"]
    print(
        f"{v['name']:<25} "
        f"{r['faithfulness_mean']:>6.2f} "
        f"{r['context_recall_mean']:>7.2f} "
        f"{r['answer_relevance_mean']:>6.2f} "
        f"{v['llm_mean']:>6.1f} "
        f"{v['adv_pass']*100:>5.0f}%"
    )

print("\nMLflow runs logged. View with: mlflow ui")

Without MLflow you're flying blind. You make a change, scores go up on a spot check, but you can't explain why the improvement happened, can't reproduce it with a different dataset version, and can't compare it against runs from three weeks ago. MLflow makes every improvement attributable — the config is the experiment. This is what "data engineering for AI" means in practice.

---
## Section 7 — Building Eval Sets from Real Sessions

Author-written test queries use vocabulary you know is in the index. Real users don't. This section shows the gap and a method to bootstrap a realistic eval set before you have real user sessions.

In [ ]:
# Formal (author-written) vs informal (user-written) vocabulary patterns
vocab_gap_examples = [
    ("What is my AAPL position?",                   "how am i doing with apple"),
    ("What is my portfolio P&L this month?",         "am i up or down this month"),
    ("What does Goldman Sachs say about NVDA?",      "what are people saying about nvidia"),
    ("What is my equity allocation by sector?",      "where is my money going"),
    ("What is my unrealized gain on MSFT?",          "how much profit would i make if i sold microsoft"),
]

print("Vocabulary gap — author queries vs real user queries:\n")
print(f"{'Formal (author-written)':<45} {'Informal (user-written)'}")
print("-" * 90)
for formal, informal in vocab_gap_examples:
    print(f"{formal:<45} {informal}")


def generate_user_style_variants(formal_query: str, n: int = 3) -> list[str]:
    """Rewrite a formal query in n casual user phrasings to bootstrap a realistic eval set."""
    raw = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=(
            "You are simulating how a retail investor — not a financial professional — "
            "would ask the same question in a casual chat interface. "
            f"Rewrite the question in {n} different informal phrasings. "
            "Use everyday language, abbreviations, and different vocabulary than the original. "
            "Return JSON only: {\"variants\": [list of strings]}"
        ),
        messages=[{"role": "user", "content": formal_query}]
    ).content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)["variants"]


seed_queries = [
    "What is my AAPL position?",
    "What is my portfolio P&L this month?",
    "What does Goldman Sachs say about NVDA?",
    "What is my tech sector concentration?",
    "Is my NVDA position at a gain or a loss?",
]

print("\n" + "=" * 70)
print("LLM-GENERATED USER-STYLE VARIANTS (bootstrapped eval set)")
print("=" * 70)
for q in seed_queries:
    variants = generate_user_style_variants(q, n=3)
    print(f"\nFormal: {q}")
    for i, v in enumerate(variants, 1):
        print(f"  {i}. {v}")

Testing on queries you write yourself tests whether the system understands your vocabulary. Real evaluation requires real user language.

Until you have real sessions, bootstrap with LLM paraphrasing: take your formal golden queries, generate 3–5 casual variants of each, and add those variants to your eval set. They exercise the vocabulary mismatch failure surface you're actually shipping into. Once you have real user sessions, replace the synthetic variants with actual queries — they'll be more diverse and more surprising than anything you generate yourself.

---
## Section 8 — Key Observations

1. **High faithfulness on a non-adversarial test set is a false signal.** Always include adversarial cases in your golden dataset. A system scoring 0.9 faithfulness without adversarial coverage may be hallucinating confidently on every out-of-scope query.

2. **Faithfulness high + context recall low = fix retrieval, not generation.** The model is being faithful to the wrong chunks. The bug is upstream. Don't touch the system prompt until retrieval is fixed.

3. **LLM-as-judge needs G-Eval chain-of-thought to reduce verbosity and position bias.** Raw scoring without a reasoning step over-weights longer answers and the first claims made. Force step-by-step reasoning before the score.

4. **MLflow makes improvements attributable.** Without it you can't explain why metrics changed or reproduce the improvement with a different dataset version. Log params, metrics, and artifacts on every eval run — the config is the experiment.

5. **Build eval sets from real user sessions, not author queries.** The vocabulary gap between how users ask and how documents are written is the most common source of false confidence in RAG evaluation. Synthetic paraphrasing via LLM is a valid bootstrap until you have real session data.

In [ ]:
print("Commit message:")
print("  04C: RAG evaluation — golden datasets, RAGAS, LLM-as-judge, MLflow tracking, adversarial pass rate")
print()
print("Push command:")
print("  git add 04-rag-systems/")
print("  git commit -m '04C: RAG evaluation — golden datasets, RAGAS, LLM-as-judge, MLflow tracking, adversarial pass rate'")
print("  git push origin main")